# အခန်း ၁၃ - Cognee Knowledge Graphs ဖြင့် Agent မှတ်ဉာဏ်


## Setup

ဤ notebook သည် [**Cognee**](https://www.cognee.ai/) ဗဟုသုတဂရပ်များနှင့် **Microsoft Agent Framework** (MAF) ကို အသုံးပြု၍ တည်ကြပ်စွာမှတ်ဉာဏ်ရှိသော အကြောင်းအရာကို သိသည့် **coding assistant** ကို ဘယ်လိုတည်ဆောက်မည်ကို ပြသသည်။

Cognee သည် unstructured စာသားများကို သယ်ဆောင်ထားသော vector embeddings ဖြင့် ကူညီသော စနစ်တကျ query ပြုလုပ်နိုင်သော ဗဟုသုတဂရပ်ဖြစ်သို့ ပြောင်းလဲပေးပြီး၊ သင်၏ agent ကို ဆက်ဆံရေးအခြေခံထားသော ရေရှည်မှတ်ဉာဏ်ကျွမ်းကျင်မှုရှိစေသည်။

### သင်ယူမည်များ
1. **ဗဟုသုတဂရပ်များ တည်ဆောက်ခြင်း**: developer profiles နှင့် အကောင်းဆုံးလေ့လာမှုများကို စနစ်တကျ query ပြုလုပ်နိုင်သော အချက်အလက်များအဖြစ် ပြောင်းလဲခြင်း။
2. **Cognee ကို MAF နှင့် ပေါင်းစပ်ခြင်း**: `@tool` function များကို သုံး၍ MAF agent သည် Cognee ၏ ဗဟုသုတဂရပ်ကို မေးမြန်းနိုင်စေခြင်း။
3. **Session-Aware ဆက်သွယ်ခြင်းများ**: တစ်ခုတည်းသော session အတွင်း မေးခွန်းများစွာတွင် context ကို ထိန်းသိမ်းခြင်း။
4. **ရေရှည်မှတ်ဉာဏ်**: session များအတွင်း အရေးကြီးသော ဗဟုသုတကို သိမ်းဆည်းထား၍ နောက်ထပ် ဆက်သွယ်မှုများတွင် ပြန်လည်ယူဆောင်နိုင်ခြင်း။

### လိုအပ်ချက်များ
- Python 3.9+
- Redis ကို ဒေသတွင်းတွင် စက်တင်ထားရန် (`docker run -d -p 6379:6379 redis`) (session စီမံခန့်ခွဲမှုအတွက်)
- LLM API key (ဥပမာ OpenAI) — `.env` တွင် `LLM_API_KEY` ကို သတ်မှတ်ထားရန်
- `.env` တွင် `CACHING=true` (Cognee session များအတွက် လိုအပ်သည်)
- Azure AI Foundry project တစ်ခု နှင့် deploy ဖြစ်ပြီးသော chat model တစ်ခု
- `.env` တွင် `AZURE_AI_PROJECT_ENDPOINT` နှင့် `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI ဖြင့် authenticated (`az login`) ပြီးဖြစ်ရမည်။


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## အေးဂျင့်မှတ်ဉာဏ်အမျိုးအစားများ

ဒီ notebook မှာ မိမိ Lesson 13 notebook မှာရှိတဲ့ မှတ်ဉာဏ်အမျိုးအစား သုံးမျိုးကို သုံးသပ်ပြီး Cognee ကို ရေရှည်မှတ်ဉာဏ် အနေနဲ့ အသုံးပြုထားပါတယ်-

| မှတ်ဉာဏ်အမျိုးအစား | မက်ခနစ် | အသက်တမ်း |
|---|---|---|
| **အလုပ်လုပ်နေဆဲ** | `agent.create_session()` (MAF) | တစ်ခါတည်းသော အပြောအဆို |
| **တိုတောင်းစေ့စပ်ရာ** | Cognee session cache (Redis) | တစ်ခါတည်းသော စက်ရှင် |
| **ရေရှည်** | Cognee knowledge graph + vectors | အမြဲတမ်း |

### Cognee ၏ မှတ်ဉာဏ် ဖွဲ့စည်းပုံ
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Cognee သိုလှောင်မှုကို ပြင်ဆင်ပါ။


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## အပိုင်း 1 — အသိပညာအခြေခံ တည်ဆောက်ခြင်း

ကျွန်ုပ်တို့၏ ကုဒ်ရေး အကူအညီသူအတွက် အသုဘ်ခြင်းရှိသော အသိပညာ အခြေခံကို ဖန်တီးရန် အမျိုးအစား သုံးမျိုးတာဝန်ယူ ထည့်သွင်းသည်-

1. **ဖွံ့ဖြိုးသူ ကိုယ်ရေးအကျဉ်း** — ကိုယ်ပိုင် ကျွမ်းကျင်မှုနှင့် နည်းပညာ မျက်မှောက်
2. **Python အကောင်းဆုံး လေ့လာမှုများ** — Python ၏ သဘောတရားနှင့် လက်တွေ့လမ်းညွှန်ချက်များ
3. **သမိုင်းကြောင်း စကားဝိုင်းများ** — ဖွံ့ဖြိုးသူများနှင့် AI အကူအညီသူများအကြား ယခင် Q&A အစည်းအဝေးများ


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## သတင်းအချက်အလက်ဂရပ်ကို မြင်သာအောင် ပြရန်

Cognee သည် ၎င်းထုတ်ယူခဲ့သည့် အရာဝတ္ထုများနှင့် ဆက်နွယ်မှုများ၏ အပြန်အလှန် HTML မြင်သာမှုကို တင်ပြနိုင်သည်။


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## မှတ်ဉာဏ်ကို Memify ဖြင့်တိုးမြှင့်ပါ

`memify()` သည် အသိပညာဂရပ်ကိုသုံးသပ်ကာ ဉာဏ်ကြီးကြပ်သော စည်းမျဉ်းများကို ဖန်တီးပေးသည် — ပုံစံများ၊ အကောင်းဆုံးလေ့လာမှုများနှင့်အတွေးအခေါ်များအကြားဆက်နွယ်မှုများကို ဖော်ထုတ်ပေးသည်။


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## အပိုင်း ၂ — Cognee Tools ပါသော MAF ကိုယ်စားလှယ်

အခုတော့ Cognee ရဲ့ knowledge graph ကို `@tool` functions ဖြင့် query လုပ်နိုင်တဲ့ MAF ကိုယ်စားလှယ်တစ်ယောက်ကို ဖန်တီးမှာပါ။ ဒါကတော့ ကိုယ်စားလှယ်ကို graph အသိပညာအခြေပြု semantic search အင်အားအပြည့်အဝကို အသုံးချစေပြီး ပွောဆိုဆက်ဆံမှု context ကို session များမှတဆင့် ထိန်းသိမ်းထားနိုင်စေပါတယ်။


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## နောက်ခံမှတ်ဉာဏ်နှင့် အစည်းအဝေးများ

`AgentSession` ( `agent.create_session()` မှ ဖန်တီးသည်) သည် အစည်းအဝေးအတွင်း နောက်ခံမှတ်ဉာဏ်ကို ပံ့ပိုးပေးသည်။ အေးဂျင့်သည် ရှေးက စကားများကို ပြန်လည်ကိုးကားနိုင်ခြင်းအပြင် Cognee ၏ ရေရှည် သိပ္ပံဇယားကိုလည်း မေးမြန်းနိုင်သည်။


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## New Session — အချိန်ကြာရှည် သတိမှတ်ဉာဏ် မသွုတ်ခေါ်

စတင်သည့် အစစ်တမ်းအသစ်သည် အလုပ်လုပ်ငန်းမှတ်ဉာဏ်ကို ဖယ်ရှားပစ်သွားသော်လည်း Cognee အတတ်ပညာဂရပ်ဖ်သည် အလွန်ရရှိနိုင်ပါသည်။ ကိုယ်စားလှယ်သည် လုံးဝအသစ်သော စကားဝိုင်းတစ်ခုတွင်တင် တူညီသောအချိန်ကြာရှည် သိသင့်သည့်အချက်များကို ပြန်လည်ရယူနိုင်သည်။


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## အနှစ်ချုပ်

ဤနုတ်ဘုတ်တွင် သင်သည် **MAF ၏ အလုပ်လုပ်သည့်မှတ်ဉာဏ်** (`agent.create_session()`) ကို **Cognee ၏ ရေရှည်သိပ္ပံပညာဇယား** နှင့်ပေါင်းစပ်ထားသည့် ကုဒ်ရေးသားသူအကူအညီကို တည်ဆောက်ခဲ့ပါသည်။

### သင်သင်ယူခဲ့သည်များ
1. **သိပ္ပံပညာဇယား ဖန်တီးခြင်း**: Cognee သည် မသိမ်းထားသည့် စာသားကို စုပ်ယူပြီး ဇယား + ဗက်တာမှတ်ဉာဏ်တည်ဆောက်သည်။
2. **memify ဖြင့် ဇယားတိုးတက်စေခြင်း**: ရရှိသော သတင်းအချက်အလက် နှင့် သင့်လက်ရှိဇယားတွင် ပိုမိုဖြစ်ထွန်းသော ဆက်နွယ်မှုများကို ထည့်သွင်းသည်။
3. **MAF + Cognee ပေါင်းစပ်ခြင်း**: `@tool` function များက MAF ကိုယ်စားလှယ်တစ်ဦးအား Cognee ၏ ဇယားကို သဘာဝကျကျ စုံစမ်းစစ်ဆေးရန် ခွင့်ပြုသည်။
4. **အလုပ်လုပ်သည့်မှတ်ဉာဏ် + ရေရှည်မှတ်ဉာဏ်**: `AgentSession` (`agent.create_session()` ဖြင့်) သည် စည်းဝေးမှုအခြေအနေကို ပေးပို့ပြီး Cognee သည် အမြဲတမ်းသိပ္ပံပညာပေးသည်။
5. **NodeSets ဖြင့် စစ်ထုတ်ရှာဖွေရေး**: သိပ္ပံပညာဇယား၏ အထူးသီးသန့်အပိုင်းများကို ရည်ရွယ်၍ ရှာဖွေနိုင်သည် (ဥပမာ- သဘောတရားများသာ)။

### အဓိကသင်ခန်းစာများ
- **Cognee** သည် မတ်တပ်ရပ်မရှိတဲ့စာသားကို စည်းအုပ်ပြီး ဆက်နွယ်မှုနှင့်အတူ မွတ်သားသော မှတ်ဉာဏ်အဖြစ် ပြောင်းလဲသွားသည် — ဗက်တာသိုလှောင်မှုတစ်ခု ထက်ပို၍ အာဏာရှိသည်။
- **`@tool` function များ**သည် MAF ကိုယ်စားလှယ်များနှင့် ပြင်ပသိပ္ပံပညာစနစ်များကို ဖောက်ထွင်းချိတ်ဆက်ပေးသည်။
- **`AgentSession`** (`agent.create_session()` မှ) သည် စကားပြောတစ်စုံတစ်ရာ့ အမျိုးအစားစကားဝိုင်းနဲ့ ရေရှည်သိပ္ပံပညာကို မတူညီသည့် သီးခြား Context နှင့် ထိန်းသိမ်းထားသည်။
- တစ်ပုံစံတည်းသော သိပ္ပံပညာဇယားကို အများအပြားသောစည်းဝေးမှုနှင့် ကိုယ်စားလှယ်များ ကိန်းသုံးသည်။

### လက်တွေ့အသုံးချနိုင်မှုများ
- **Developer copilots**: ကုဒ်ပြန်လည်ဆန်းစစ်ခြင်း၊ တာဝန်ချထောက်ခံရေး၊ အဆောက်အအုံအကူအညီများ
- **Customer-facing copilots**: ထုတ်ကုန်စာရွက်စာတမ်းများ၊ အမေးအဖြေများနှင့် CRM မှတ်စုများ ပေါ်တွင် ဝန်ဆောင်မှုအကူအညီများ
- **Internal expert copilots**: မူဝါဒ၊ ဥပဒေ သို့မဟုတ် လုံခြုံရေးအကူအညီများ - လမ်းညွှန်ချက်များ စဉ်းစားခင်းကျင်းရန်
- **တစ်ကြောင့်ညီသော ဒေတာအလွှာများ**: စနစ်တကျတည်းဖြတ်ထားသောနှင့် မသိမ်းထားသော ဒေတာများကို တစ်ခုသော စုံစမ်းမေးခွန်းပိုင်း ဇယားအဖြစ် ပေါင်းစပ်ခြင်း

### နောက်တစ်ခြားလုပ်ဆောင်ရန်များ
- Cognee တွင် အချိန်အခြေအနေကို ကြိုးစားစမ်းသပ်ခြင်း
- ဒိုမိန်းအချက်အလက်ဖြင့် OWL ontology တည်ဆောက်ခြင်း
- ရှာဖွေမှုတိုးတက်စေရန် အသုံးပြုသူတုံ့ပြန်ချက် ဖန်တီးခြင်း
- တူညီသော Cognee မှတ်ဉာဏ်အလွှာကို မတူညီသော ကိုယ်စားလှယ်စနစ်များဖြင့် တိုးချဲ့မှု


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ကြေညာချက်**  
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှုဖြစ်သော [Co-op Translator](https://github.com/Azure/co-op-translator) ကနေ ဘာသာပြန်ထားခြင်းဖြစ်ပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက်ကြိုးစားသော်လည်း အလိုအလျောက် ဘာသာပြန်ချက်တွင် အမှားများ သို့မဟုတ် မှားယွင်းမှုများ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် တောင်းဆိုအပ်ပါသည်။ မူရင်းစာတမ်းကို မူလဘာသာဖြင့်သာ အတည်ပြုရသော အရင်းခံအချက်အလက်အဖြစ် သတ်မှတ်သင့်ပါသည်။ အရေးကြီးသော အချက်အလက်များအတွက်လျှင် အတည်ပြုနိုင်သော လူ့ဘာသာပြန်သူအကူအညီရယူရန် အကြံပြုလိုက်ပါသည်။ ဤဘာသာပြန်ချက် အသုံးပြုမှုကြောင့် ဖြစ်ပေါ်လာနိုင်သည့် အနားလွှဲနားမလည်မှုများ သို့မဟုတ် မှားအတွင်းလိုက်မှုများအတွက် ကျွန်ုပ်တို့သည် တာဝန်မပေးပန်ပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
